In [16]:
#Célula 1 — imports e configuração

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import boto3
import pandas as pd

# Caminho local onde estão os snapshots RAW.
# Ajuste este caminho se sua pasta estiver em outro local.
RAW_LOCAL_PATH = Path("/app/data/raw_local/RAW")

# Configurações do MinIO.
# Dentro da rede Docker, usamos o hostname "minio", não "localhost".
MINIO_ENDPOINT = "http://minio:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "admin12345"
BUCKET_NAME = "contracts"

# Prefixo lógico da Landing Zone dentro do bucket.
LANDING_PREFIX = "landing"

In [17]:
#Célula 2 — Cliente S3/MinIO

# Cria um cliente S3 compatível com MinIO.
# boto3 é usado aqui porque a Landing trabalha com arquivos brutos/binários.
s3_client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1",
)

# Garante que o bucket existe antes de enviar arquivos.
existing_buckets = [
    bucket["Name"]
    for bucket in s3_client.list_buckets()["Buckets"]
]

if BUCKET_NAME not in existing_buckets:
    s3_client.create_bucket(Bucket=BUCKET_NAME)

print(f"Bucket disponível: {BUCKET_NAME}")

Bucket disponível: contracts


In [18]:
#Célula 3 — Função de hash

def calculate_file_hash(file_path: Path) -> str:
    """
    Calcula o hash SHA-256 do conteúdo do arquivo.

    Por que SHA-256?
    - Permite identificar se o conteúdo mudou.
    - Evita depender apenas de nome de arquivo.
    - Ajuda em auditoria e deduplicação.
    """
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        for block in iter(lambda: file.read(8192), b""):
            sha256.update(block)

    return sha256.hexdigest()

In [19]:
#Célula 4 — Descobrir arquivos locais

def discover_raw_files(raw_path: Path) -> pd.DataFrame:
    """
    Descobre todos os arquivos dentro da pasta RAW local.

    A função assume que a primeira pasta abaixo de RAW representa o snapshot.
    Exemplo:
      RAW/2026-06-13_0800/RELATORIO/.../arquivo.csv
    """
    records = []

    for file_path in raw_path.rglob("*"):
        # Ignora diretórios. Só processamos arquivos.
        if not file_path.is_file():
            continue

        # Caminho relativo a partir da pasta RAW.
        relative_path = file_path.relative_to(raw_path)

        # A primeira parte do caminho é o snapshot.
        snapshot_date = relative_path.parts[0]

        # Caminho destino dentro do MinIO.
        landing_key = (
            f"{LANDING_PREFIX}/"
            f"snapshot_date={snapshot_date}/"
            f"{'/'.join(relative_path.parts[1:])}"
        )

        records.append(
            {
                "snapshot_date": snapshot_date,
                "source_file": file_path.name,
                "source_path": str(file_path),
                "relative_path": str(relative_path),
                "landing_bucket": BUCKET_NAME,
                "landing_key": landing_key,
                "file_extension": file_path.suffix.lower(),
                "file_size_bytes": file_path.stat().st_size,
                "file_hash": calculate_file_hash(file_path),
                "discovered_at": datetime.now(timezone.utc).isoformat(),
            }
        )

    return pd.DataFrame(records)


files_df = discover_raw_files(RAW_LOCAL_PATH)

print(f"Arquivos encontrados: {len(files_df)}")
files_df.head()

Arquivos encontrados: 36


,snapshot_date,source_file,source_path,relative_path,landing_bucket,landing_key,file_extension,file_size_bytes,file_hash,discovered_at
0,2024-07-13_0800,ControleMedicoesPagamentos.csv,/app/data/raw_local/RAW/2024-07-13_0800/CONTRO...,2024-07-13_0800/CONTROLE DE MEDICOES E PAGAMEN...,contracts,landing/snapshot_date=2024-07-13_0800/CONTROLE...,.csv,39634,b73d09f9dc6766eb918f3e1149649f758e0338d0f42f86...,2026-06-22T03:41:20.926442+00:00
1,2024-07-13_0800,Exportação_bm_acompanhamento.xlsx,/app/data/raw_local/RAW/2024-07-13_0800/CONTRO...,2024-07-13_0800/CONTROLE DE MEDICOES EM ANDAME...,contracts,landing/snapshot_date=2024-07-13_0800/CONTROLE...,.xlsx,18912,84df6ac3cb58f7ca789bf83e192ad99369bb1d35c6b41f...,2026-06-22T03:41:20.948990+00:00
2,2024-07-13_0800,202211_ADMIN.xlsb,/app/data/raw_local/RAW/2024-07-13_0800/NACT/2...,2024-07-13_0800/NACT/202211_ADMIN.xlsb,contracts,landing/snapshot_date=2024-07-13_0800/NACT/202...,.xlsb,20251932,5d05acde002b287d9068b5b4b8dc34dc04609d1cc16cbb...,2026-06-22T03:41:21.477985+00:00
3,2024-07-13_0800,Pendências-010223.xlsx,/app/data/raw_local/RAW/2024-07-13_0800/PENDEN...,2024-07-13_0800/PENDENCIAS RDO/Pendências-0102...,contracts,landing/snapshot_date=2024-07-13_0800/PENDENCI...,.xlsx,23053,3cca4c48eaa038cee8bd806f6cae89ca1de1e6ee0e9996...,2026-06-22T03:41:21.495983+00:00
4,2024-07-13_0800,QEC_5900055119_7_55_77.csv,/app/data/raw_local/RAW/2024-07-13_0800/QUADRO...,2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/Q...,contracts,landing/snapshot_date=2024-07-13_0800/QUADRO E...,.csv,129525,f8a0265c96ffd7085b4e65eff75c24ba5d3439daeabbc2...,2026-06-22T03:41:21.516254+00:00


In [20]:
#Célula 5 — Enviar arquivos para o MinIO

def upload_file_to_landing(row: pd.Series) -> dict:
    """
    Envia um arquivo local para a Landing Zone no MinIO.

    Regra importante:
    - A Landing preserva o arquivo original.
    - Nenhuma transformação de conteúdo é feita aqui.
    """
    source_path = Path(row["source_path"])
    landing_key = row["landing_key"]

    s3_client.upload_file(
        Filename=str(source_path),
        Bucket=BUCKET_NAME,
        Key=landing_key,
    )

    return {
        **row.to_dict(),
        "upload_status": "SUCCESS",
        "uploaded_at": datetime.now(timezone.utc).isoformat(),
    }


upload_results = []

for _, row in files_df.iterrows():
    try:
        result = upload_file_to_landing(row)
        upload_results.append(result)

    except Exception as error:
        # Em projeto real, esse erro também iria para log estruturado.
        failed_result = row.to_dict()
        failed_result["upload_status"] = "FAILED"
        failed_result["error_message"] = str(error)
        failed_result["uploaded_at"] = datetime.now(timezone.utc).isoformat()

        upload_results.append(failed_result)

landing_metadata_df = pd.DataFrame(upload_results)

landing_metadata_df["upload_status"].value_counts()


upload_status
SUCCESS    36
Name: count, dtype: int64

In [21]:
#Célula 6 — Validar objetos no MinIO

# Lista objetos enviados para confirmar que a Landing foi populada.
response = s3_client.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=f"{LANDING_PREFIX}/",
)

objects = response.get("Contents", [])

print(f"Objetos encontrados na Landing: {len(objects)}")

for obj in objects[:10]:
    print(obj["Key"])

Objetos encontrados na Landing: 36
landing/snapshot_date=2024-07-13_0800/CONTROLE DE MEDICOES E PAGAMENTOS/ControleMedicoesPagamentos.csv
landing/snapshot_date=2024-07-13_0800/CONTROLE DE MEDICOES EM ANDAMENTO/Exportação_bm_acompanhamento.xlsx
landing/snapshot_date=2024-07-13_0800/NACT/202211_ADMIN.xlsb
landing/snapshot_date=2024-07-13_0800/PENDENCIAS RDO/Pendências-010223.xlsx
landing/snapshot_date=2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/QEC_5900055119_7_55_77.csv
landing/snapshot_date=2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/QEC_5900074722_7_55_77.csv
landing/snapshot_date=2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/QEC_5900083950_7_55_77.csv
landing/snapshot_date=2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/QEC_5900086165_7_55_77.csv
landing/snapshot_date=2024-07-13_0800/QUADRO EVOLUTIVO DO CONTRATO/QEC_5900086169_7_55_77.csv
landing/snapshot_date=2024-07-13_0800/RELATORIO ANALITICO DO PROJETO/AnaliticoProjeto.csv


In [22]:
#Célula 7 — Salvar metadados da Landing

# Salva metadados localmente para auditoria inicial.
# Depois podemos evoluir isso para DuckDB ou uma tabela metadata no próprio lakehouse.
metadata_output_path = Path("/app/data/landing_metadata.csv")

landing_metadata_df.to_csv(
    metadata_output_path,
    index=False,
    encoding="utf-8",
)

print(f"Metadados salvos em: {metadata_output_path}")

Metadados salvos em: /app/data/landing_metadata.csv


In [23]:
from src.landing.loader import run_landing_pipeline

metadata_df = run_landing_pipeline()

metadata_df["upload_status"].value_counts()

upload_status
SUCCESS    36
Name: count, dtype: int64